# Leak-free N=24 baseline, and the N=48 data control

Two finished runs:

| job | elapsed | what it did |
|---|---|---|
| **49659502** | 05:41:35 | retrained the 0.9960 record with `--shuffle-adj`, then re-ran the shared pool, the ordering counterfactual, the memorization gap and a depth breakdown on it |
| **49639313** | 03:24:52 | N=48 at 80000 instances for L=3 and L=4, closing the data-budget confound on L*(48)=4 |

Both ExitCode 0:0.

## Provenance

* **Training commit** `5103e83` (the commit that added `SudokuReasoningConfig.shuffle_adj`); **analysis/package commit** `b0baa00`.
* **Seed** 0 for training in both runs. Instance streams are seed-offset: training from `seed`, the shared pool from `seed_start=50_000_000_000`, the adjacency permutation from a dedicated stream at offset 40,000,000.
* **Config, 49659502:** N=24, degree=5, q=4, rho=0.3, rho_max=0.95, `hint_mode=removal`, d_model=256, nhead=4, L=12, ff=1024, pre-norm, lr=1e-3 constant, batch=32, 300 epochs, n_train=80000, n_val=300, **`shuffle_adj=True`**. Identical to job 48912548 (the 0.9960 record) in every other field.
* **Config, 49639313:** N=48, degree=5, q=4, same optimizer settings, L in {3, 4}, n_train=80000, `shuffle_adj=False` (it is a continuation of the leaked-order layer scan, so its numbers are decoding scores and are *not* comparable to the leak-free cell above).
* **Metric of record:** `sudoku_agg_eval` shared pool, n=1500, instance *i* at `seed_start+i`, Wilson 95% intervals, paired McNemar exact between cells on the same pool. `shuffle_adj` is in `INSTANCE_KEYS`, so a `_shuf` cell shares a pool only with other `_shuf` cells.

In [ ]:
import os, csv, json
import numpy as np
import pandas as pd

SCR  = os.environ["SCRATCH"] + "/spin_gnn"
LF   = SCR + "/sudoku_n24_leakfree_2026-10-26"
LEAK = SCR + "/sudoku_memorization_2026-10-26"
N48  = SCR + "/sudoku_n48_data_control_2026-10-26"

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1. Headline: the leak-free baseline, against the previous best on the same eval

The previous best is job 48912548 at **0.9960** on this protocol. That number is not wrong, it is just not a solving score: job 49655447 showed the model is reading the neighbour ordering, which `planted_regular_graph` emitted sorted by the neighbour's true colour for 100.0% of nodes (finding 16).

The two cells are measured on *different* pools by construction — `shuffle_adj` changes the instances — so this is a comparison of two protocols, not a paired test, and only the Wilson intervals are meaningful.

In [ ]:
lf_pool = pd.read_csv(LF + "/sharedpool_n1500.csv")
print(lf_pool.to_string(index=False))

row = lf_pool.iloc[0]
print(f"\nleak-free : {row.solve_rate:.4f}  [{row.wilson_lo:.4f}, {row.wilson_hi:.4f}]"
      f"   (n={int(row.n_instances)})")
print( "previous  : 0.9960  [0.9918, 0.9981]   (n=1500, job 48912548, colour-sorted pool)")
print(f"\ndelta of protocols: {row.solve_rate - 0.9960:+.4f}")

## 2. The ordering counterfactual: is the leak-free model actually order-insensitive?

Each ordering permutes `adj[v]` while preserving the edge set, the colouring, the hints and the DD trace — the trace is recomputed on the reordered adjacency and asserted equal, so only the token order changes.

* `identity` — as trained
* `within_block` — permute inside each colour block, so the colour *code* survives but the order within it does not
* `shuffle` — uniformly random
* `byid` — sorted by node id, a different deterministic order with no colour information
* `bycolor` — sorted by the neighbour's true colour, i.e. **the leak handed back**

`bycolor` is the decisive control. If the model wanted the shortcut, giving it back would help.

In [ ]:
om = pd.read_csv(LF + "/ordering_and_memorization.csv")
orderings = om[om.part == "ordering"][
    ["ordering", "heldout_solve_rate", "heldout_wilson_lo", "heldout_wilson_hi", "p", "n_instances"]
]
print("leak-free model (job 49659502), n=400, paired McNemar exact vs identity")
print(orderings.to_string(index=False))

prev = pd.read_csv(LEAK + "/ordering_counterfactual.csv")
prev = prev[prev.n_nodes == 24][["ordering", "heldout_solve_rate", "p"]]
print("\nthe 0.9960 record (job 49655447), same protocol")
print(prev.to_string(index=False))

The record falls 0.998 → 0.145 under `shuffle` and → 0.083 under `byid`, while `within_block` ties `identity` at p=1: it is reading the colour code, not the order per se. The leak-free model moves by at most 0.015 across all five, including `bycolor` at −0.0100 (p=0.54).

**This settles the expensive question.** The model does not reach for the shortcut when it is offered, so the *encoding* needs no change — a set-based or permutation-invariant adjacency representation is not required. Only the generator had to change, and it already has.

## 3. Depth now binds

This is the new scientific content. Under the leak, solve_rate was flat in DD depth T (finding 11 measured L*(T) flat in T, on a task where depth did not bind). Leak-free, at **fixed realized hint density**, it falls monotonically.

The realized-density column is the control: at rho=0.75 it is identical (0.7500) in all three bins, so the fall is not a density confound. `truncated_rate` is reported so a generation-length artefact would be visible rather than silent.

In [ ]:
dep = pd.read_csv(LF + "/depth_n24_leakfree.csv")
print(dep.to_string(index=False))

for rho, g in dep.groupby("rho_requested"):
    r = np.corrcoef(g["T"], g["solve_rate"])[0, 1]
    print(f"\nrho={rho}:  solve_rate {g.solve_rate.iloc[0]:.3f} -> {g.solve_rate.iloc[-1]:.3f} "
          f"over T={g['T'].iloc[0]}..{g['T'].iloc[-1]},  r(T, solve_rate) = {r:+.3f},  "
          f"realized_rho spread = {g.realized_rho.max() - g.realized_rho.min():.4f}")

## 4. Memorization: pre-registration (d), falsified

I predicted **no** gap at n_train=80000, replicating the leaked N=96 @80k result (−0.0133, p=0.355). Measured: **+0.1000 at p=2.6e-09**.

The training side is scored on a recoverable prefix of the model's own training set (index *i* is `build_instances(cfg, 1, i)`), the held-out side on fresh seeds; unpaired two-proportion z-test. Removing the shortcut makes the task harder, so the same 80000 instances no longer cover it — the obverse of "more data closes the gap", and a direct argument that the leak-free runs are **not data-saturated at 80000**.

In [ ]:
cols = ["train_solve_rate", "heldout_solve_rate", "gap", "p", "n_instances"]
mem_lf = om[om.part == "memorization"][cols].assign(run="N=24 L=12 @80k  shuffle_adj")
mem_leak = pd.read_csv(LEAK + "/memorization_gap.csv")
# n_train_instances is blank in the rows written before 38d503a started recording it;
# those cells were the 20000-instance default.
nt = mem_leak.n_train_instances.fillna(20000).astype(int)
mem_leak = mem_leak.assign(
    run=[f"N={int(r.n_nodes)} L={int(r.num_layers)} @{n // 1000}k  leaked"
         for r, n in zip(mem_leak.itertuples(), nt)])[cols + ["run"]]
print(pd.concat([mem_leak, mem_lf])[["run"] + cols].to_string(index=False))

## 5. Second run (49639313): 4x data at N=48 does not move the knee

L*(48)=4 was measured at n_train=20000 and could have been a data-budget artefact rather than a depth requirement. Pre-registered: (a) L=3 @80k still misses at least one rho=0.60 bin, (b) L=4 @80k improves, (c) the 4x gain at N=48 is smaller than the +0.1007 seen at N=96. All three confirmed.

In [ ]:
p48 = pd.read_csv(N48 + "/sharedpool_n1500.csv")
print(p48[["checkpoint", "n_instances", "solve_rate", "wilson_lo", "wilson_hi",
           "delta", "discordant_won", "discordant_lost", "mcnemar_exact_p"]].to_string(index=False))

d48 = pd.read_csv(N48 + "/depth_by_layers_n48_80k.csv")
print("\nper-depth at 80000:")
print(d48[["checkpoint", "rho_requested", "T", "n", "realized_rho", "solve_rate", "truncated_rate"]]
      .to_string(index=False))

for ck, g in d48.groupby("checkpoint"):
    for rho, gg in g.groupby("rho_requested"):
        r = np.corrcoef(gg["T"], gg["solve_rate"])[0, 1]
        print(f"{ck}  rho={rho}:  r(T, solve_rate) = {r:+.3f}")

**Not predicted, and unexplained:** 4x data *costs* the under-depth model L=3 a quarter of its solve_rate (0.8293 → 0.6560, 32–412 discordant, p=3.0e-85). The pool is held out, so it is not pool memorization. Note also that L=3's solve_rate *rises* with T (r=+0.92 at rho=0.75, +0.83 at rho=0.60) — it does better on deeper instances, which no implementation of DD can do, so whatever L=3 is doing is not a truncated DD. This is recorded as an open problem, not given a mechanism.

## 6. Plots

The slide deck is built by `python -m notebooks.make_leakfree_deck` and written to `notebooks/sudoku_leakfree_2026-10-26/leakfree_baseline.pdf` (6 landscape pages). The cell below regenerates it.

In [ ]:
from notebooks.make_leakfree_deck import main as build_deck
build_deck()

## 7. What this changes, and what is still owed

* **The clean N=24 baseline is 0.8540, not 0.9960.** Every other solve_rate currently in STATUS.md is a decoding score.
* **Re-measuring is retraining.** `shuffle_adj` is in `sudoku_agg_eval.INSTANCE_KEYS`, so no `_shuf` cell pools with any existing checkpoint. There is no way to convert the old numbers; the cells that matter have to be run again.
* **The leak-free L\*(T) series at N=24 is now the highest-value run**, because depth finally binds (§3) — it is the measurement finding 11 was trying to be. Scope: L in {2,3,4,6,8} at 80000+ instances, `_shuf`, ~5 cells.
* **Open:** why 4x data costs L=3 0.25 solve_rate at N=48 (§5).
* **Running:** job 49702811 runs the SAE on this checkpoint against the leaked record, with edge/justifier predicates, a position-rolled null control and an aliveness floor, to ask whether the leak-free model represents the relevant nodes and edges at the relevant times.